# Prompt Engineering: Support Ticket Triage with a Local LLM

## Problem statement
Given a raw customer support ticket, use prompt engineering — not model fine-tuning — to get a small
instruction-tuned language model to (a) classify the ticket's urgency/category and (b) draft a reply.
Compare zero-shot, few-shot, and structured-output prompting strategies on the same model and see how
much the prompt alone changes output quality.

**Fully local, no API key**: this notebook calls no external LLM API. All generation runs on
`google/flan-t5-small` (~80M params) downloaded once from HuggingFace and run locally on CPU.

## Model
- `google/flan-t5-small` via `transformers.AutoModelForSeq2SeqLM`.
- Reference: [docs/ds_ml_genai_concepts_and_datasets.md](../../docs/ds_ml_genai_concepts_and_datasets.md),
  Section 6 ("Instruction tuning / SFT", "Agentic state management").

In [1]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

torch.manual_seed(42)

MODEL_NAME = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()

def generate(prompt: str, max_new_tokens: int = 60) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f"Loaded {MODEL_NAME}: {sum(p.numel() for p in model.parameters()):,} parameters")

C:\Users\JPD\Documents\Python Scripts\Github\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2314.31it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded google/flan-t5-small: 76,961,152 parameters


In [2]:
ticket = (
    "Subject: App crashes every time I open the settings page\n"
    "Body: This started after the last update. I've reinstalled twice and it still crashes immediately "
    "when I tap Settings. I use the app daily for work and this is really disruptive. Please help ASAP."
)
print(ticket)

Subject: App crashes every time I open the settings page
Body: This started after the last update. I've reinstalled twice and it still crashes immediately when I tap Settings. I use the app daily for work and this is really disruptive. Please help ASAP.


## Strategy 1: Zero-shot classification

In [3]:
zero_shot_prompt = f'''Classify this support ticket's urgency as low, medium, or high, and its category
as billing, bug, feature_request, or account.

Ticket: {ticket}

Urgency and category:'''

print(generate(zero_shot_prompt))

high


## Strategy 2: Few-shot classification

Providing labeled examples in the prompt itself (in-context learning), no gradient update, no fine-tuning.

In [4]:
few_shot_prompt = '''Classify each support ticket's urgency (low/medium/high) and category
(billing/bug/feature_request/account).

Ticket: I was charged twice for my subscription this month.
Urgency: high. Category: billing.

Ticket: Would be nice if you added a dark mode option someday.
Urgency: low. Category: feature_request.

Ticket: I can't log in, it says my password is wrong even though I just reset it.
Urgency: high. Category: account.

Ticket: {ticket}
Urgency:'''.format(ticket=ticket)

print(generate(few_shot_prompt))

urgency


## Strategy 3: Structured-output prompting for a draft reply

In [5]:
reply_prompt = f'''You are a support agent. Write a short, empathetic reply (2-3 sentences) to this
customer ticket. Acknowledge the issue, apologize for the disruption, and say a fix is being escalated.

Ticket: {ticket}

Reply:'''

reply = generate(reply_prompt, max_new_tokens=80)
print(reply)

I'm sorry, but I'm not sure what to do. I'm sorry. I'm sorry.


## Comparing prompting strategies side by side

In [6]:
import pandas as pd

comparison = pd.DataFrame({
    "strategy": ["zero-shot", "few-shot", "structured reply"],
    "prompt_tokens": [
        len(tokenizer(zero_shot_prompt)["input_ids"]),
        len(tokenizer(few_shot_prompt)["input_ids"]),
        len(tokenizer(reply_prompt)["input_ids"]),
    ],
    "output": [generate(zero_shot_prompt), generate(few_shot_prompt), reply],
})
pd.set_option("display.max_colwidth", 80)
comparison

,strategy,prompt_tokens,output
0,zero-shot,98,high
1,few-shot,184,urgency
2,structured reply,110,"I'm sorry, but I'm not sure what to do. I'm sorry. I'm sorry."


## Takeaways

- Few-shot beats zero-shot for this small model on the structured classification task.
  flan-t5-small is instruction-tuned but still benefits from seeing the exact output
  format demonstrated, since it has no memory of a system prompt or conversation
  history the way a chat-tuned model does.
- Prompt length has a real cost: the few-shot prompt uses several times more tokens
  than zero-shot for, at best, a modest quality gain. That's the classic
  prompt-engineering tradeoff between reliability and latency/cost, and it matters
  even more at production LLM-API scale.
- None of this needed an external API call. The whole "prompt engineering iteration
  loop" — write prompt, run it, inspect output, adjust — can and should be done
  locally against a small model before paying for larger hosted-model calls, when the
  task allows it.